In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:

load_dotenv()

RAW_DIR = Path(os.getenv("STUDENT_RAW_DIR")).expanduser()
OUT_DIR = Path(os.getenv("OUTPUTS_DIR")).expanduser()

for p in (RAW_DIR, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("RAW_DIR :", RAW_DIR)
print("OUT_DIR :", OUT_DIR)

In [ ]:
raw_csvs = sorted(RAW_DIR.rglob("*.csv"))
print(f"Found {len(raw_csvs)} CSV files")
for p in raw_csvs[:10]:  # preview
    print(" -", p.relative_to(RAW_DIR))

In [ ]:
# Schema check: defensive loader for many CSVs at once. 
# (1) handle tricky text encodings, 
# (2) walk your RAW_DIR recursively, 
# (3) keep going if one file fails, and 
# (4) gives a clean dict of DataFrames to use right away.


def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    """
    Read CSV with sensible defaults.
    Add parse_dates=['col1', ...] or dtype={'col': 'string'} via kwargs if needed.
    """
    try:
        return pd.read_csv(path, encoding="utf-8-sig", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kwargs)

def load_all_csvs(base: Path, **kwargs) -> dict[str, pd.DataFrame]:
    """
    Load ALL CSVs under base (recursively) into a dict keyed by relative path (without .csv).
    """
    tables, errors = {}, []
    for p in sorted(base.rglob("*.csv")):
        key = str(p.relative_to(base).with_suffix(""))  # e.g. "sub/filename"
        try:
            tables[key] = read_csv_smart(p, **kwargs)
        except Exception as e:
            errors.append((str(p), str(e)))
    print(f"Loaded {len(tables)} tables; errors: {len(errors)}")
    if errors:
        for name, err in errors[:10]:
            print(" -", name, "->", err)
    return tables

tables = load_all_csvs(
    RAW_DIR,
    sep=None,        # auto-detect delimiter
    engine="python", # required for sep=None sniffing
)

# Example: tables["filename"] or tables["sub/folder/filename"]


In [ ]:
# show summary of data files
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_seq_items", None)   # don't truncate lists

summary = (
    pd.DataFrame(
        [{
            "key": k,
            "rows": len(df),
            "cols": df.shape[1],
            "columns": list(map(str, df.columns))  # full list, no manual truncation
        } for k, df in tables.items()]
    ).sort_values("key")
)

summary


In [ ]:
# For now, Continue with data files that have above 40,000 rows

# Step 1: Map clean DataFrame names -> source keys in `tables`
custom_names = {
    "sdo1_characteristics": "sdo1-student_characteristics",
    "sdo2_skc": "sdo2-skc",
    "sdo5_degree": "sdo5-student_degree_drop-out_postalcode",
    "sdo6_course_results": "sdo6-course_results"
}

# Step 2: Create DataFrames with clean names
for clean_name, source_key in custom_names.items():
    df = tables.get(source_key)
    if df is None:
        print(f"[warning] '{source_key}' not found in tables.")
        continue
    if len(df) > 40000:
        globals()[clean_name] = df
        print(f"Created DataFrame: {clean_name} from '{source_key}' with {len(df)} rows")
    else:
        print(f"Skipped '{source_key}' (only {len(df)} rows)")


In [ ]:
sdo6_course_results 

# Data Overview: schema and Sanity check
Verifying that each dataset loaded matches the structure (“schema”) as expected

In [ ]:
# Using the CLEAN names defined on the LEFT side of `custom_names`
clean_names = list(custom_names.keys())

# 1) Gather DataFrames by clean names
df_registry = {}
for name in clean_names:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        df_registry[name] = globals()[name]
    else:
        print(f"[warning] DataFrame '{name}' not found. Skipping.")

# 2) Dynamic schema/profile (fully inferred)
schema_rows = []
column_rows = []

for df_name, df in df_registry.items():
    n_rows = len(df)
    n_cols = df.shape[1]

    # fast column-level stats
    nunique = df.nunique(dropna=True)
    null_any = df.isna().any()

    # candidate single-column PKs: no nulls & unique == rowcount
    candidate_single_pk = [c for c in df.columns if (not null_any[c]) and (nunique[c] == n_rows)]

    schema_rows.append({
        "dataset": df_name,
        "rows": int(n_rows),
        "cols": int(n_cols),
        "candidate_single_primary_keys": candidate_single_pk if candidate_single_pk else None,
    })

    for col in df.columns:
        s = df[col]
        dtype_str = str(s.dtype)
        nulls = int(s.isna().sum())
        non_nulls = int(n_rows - nulls)
        nunq = int(nunique[col])
        sample_non_null = s.dropna().head(3).tolist()

        info = {
            "dataset": df_name,
            "column": str(col),
            "dtype": dtype_str,
            "nulls": nulls,
            "non_nulls": non_nulls,
            "null_pct": (nulls / n_rows * 100.0) if n_rows else np.nan,
            "unique_values": nunq,
            "example_values": sample_non_null,
        }

        # min/max for numeric or datetime-like
        if np.issubdtype(s.dtype, np.number):
            info["min"] = pd.to_numeric(s, errors="coerce").min()
            info["max"] = pd.to_numeric(s, errors="coerce").max()
        elif np.issubdtype(s.dtype, np.datetime64):
            s_dt = pd.to_datetime(s, errors="coerce")
            info["min"] = s_dt.min()
            info["max"] = s_dt.max()
        else:
            info["min"] = None
            info["max"] = None

        column_rows.append(info)

# 3) Reports
schema_report = pd.DataFrame(schema_rows).sort_values("dataset").reset_index(drop=True)
column_profile = pd.DataFrame(column_rows).sort_values(["dataset", "column"]).reset_index(drop=True)

# Show high-level first; 'column_profile' is available for deeper inspection
schema_report


------------------- Check for duplicate SINH_ID, other columns and in rows---------------------------------------

In [ ]:
# Checks each dataset for duplicate primary keys, handling key variants ('SINH_ID' / 'sinh_id' / 'sinh_d').
# Prints a per-dataset summary (rows, unique IDs, duplicate rows/IDs) and previews the most frequent duplicates.
# If duplicates exist, automatically deduplicates on the key using KEEP_POLICY (default: keep='last') and verifies the result.
# Stores all removed duplicate rows in `removed_dups` for auditability; keeps cleaned DataFrames in `datasets`.
# Reassigns the cleaned DataFrames back to the original variables (sdo1_characteristics, sdo2_skc, sdo5_degree, sdo6_course_results).
# Tip: change KEEP_POLICY to 'first' if that better matches your convention; add/remove datasets in the `datasets` dict as needed.


def find_key_col(df: pd.DataFrame) -> str:
    """Find the key column (handles SINH_ID / sinh_id / sinh_d)."""
    cmap = {c.lower(): c for c in df.columns}
    for cand in ("sinh_id", "sinh_d"):
        if cand in cmap:
            return cmap[cand]
    raise KeyError("No SINH_ID/sinh_id/sinh_d column found.")

def dup_summary(df: pd.DataFrame, key: str) -> dict:
    return {
        "rows": len(df),
        "unique_ids": df[key].nunique(dropna=True),
        "dup_rows": int(df.duplicated(subset=[key], keep=False).sum()),
        "dup_ids": int((df[key].value_counts(dropna=True) > 1).sum()),
    }

def dedup_df(df: pd.DataFrame, key: str, keep: str = "last"):
    """Return (clean_df, dropped_rows_df) by deduplicating on key."""
    mask_dups_to_drop = df.duplicated(subset=[key], keep=keep)
    dropped = df.loc[mask_dups_to_drop].copy()
    clean = df.drop_duplicates(subset=[key], keep=keep)
    return clean, dropped

datasets = {
    "sdo1_characteristics": sdo1_characteristics,
    "sdo2_skc": sdo2_skc,
    "sdo5_degree": sdo5_degree,
    "sdo6_course_results": sdo6_course_results,
}

removed_dups = {}   # keep dropped rows for audit
KEEP_POLICY = "last"  # change to "first" if preferred

for name, df in datasets.items():
    key = find_key_col(df)
    s = dup_summary(df, key)
    print(f"\n{name}: key='{key}' | rows={s['rows']:,} | unique_ids={s['unique_ids']:,} | "
          f"duplicate_rows={s['dup_rows']:,} | duplicate_ids={s['dup_ids']:,}")

    # show a quick peek of top duplicate IDs (if any)
    if s["dup_ids"] > 0:
        top_dups = (
            df.groupby(key, dropna=False).size()
              .reset_index(name="count")
              .query("count > 1")
              .sort_values("count", ascending=False)
              .head(10)
        )
        print("Top duplicate IDs:")
        print(top_dups.to_string(index=False))

    # ---- IF condition to ensure duplicates are handled ----
    if s["dup_ids"] > 0:
        print(f"-> Deduplicating {name} on '{key}' (keep='{KEEP_POLICY}')")
        clean, dropped = dedup_df(df, key, keep=KEEP_POLICY)
        datasets[name] = clean
        removed_dups[name] = dropped
        # Post-check
        s2 = dup_summary(clean, key)
        print(f"   After dedup: rows={s2['rows']:,} | unique_ids={s2['unique_ids']:,} | "
              f"duplicate_rows={s2['dup_rows']:,} | duplicate_ids={s2['dup_ids']:,}")
    else:
        print("-> No duplicates found; no action taken.")

# (Optional) reassign cleaned dataframes back to your variables
sdo1_characteristics = datasets["sdo1_characteristics"]
sdo2_skc             = datasets["sdo2_skc"]
sdo5_degree          = datasets["sdo5_degree"]
sdo6_course_results  = datasets["sdo6_course_results"]

# (Optional) inspect what was removed for any dataset:
# removed_dups["sdo6_course_results"].head()



-------------------------------- Inspect what was loaded ----------------------------------------------------------------------------

In [ ]:
# Top 50 most-null columns across all datasets
column_profile.sort_values("null_pct", ascending=False).head(50)


In [ ]:
# sdo1_characteristics dataset columns
column_profile[column_profile["dataset"] == "sdo1_characteristics"]

In [ ]:
# sdo2_skc dataset columns
column_profile[column_profile["dataset"] == "sdo2_skc"]

In [ ]:
# sdo5_degree dataset columns
column_profile[column_profile["dataset"] == "sdo5_degree"]

In [ ]:
# sdo6_course_results dataset columns
column_profile[column_profile["dataset"] == "sdo6_course_results"]

------------------------ Standardize columns before combining datasets------------------------------------------------------------

In [ ]:
# - Standardized primary key names by renaming 'sinh_d'/'sinh_id' to 'SINH_ID'.
# - Normalized column headers: collapsed whitespace into underscores
#   (e.g., "Total Credits Block A" → "Total_Credits_Block_A").
# - Preserved provenance by prefixing all non-key columns with the dataset prefix
#   (sdo1_, sdo2_, sdo5_, sdo6_); 'SINH_ID' is not prefixed.
# - Merged sdo1_characteristics, sdo2_skc, sdo5_degree, sdo6_course_results on SINH_ID
#   via left joins (sdo1 as base) to produce final DataFrame `data`.
# - Note: if any input has duplicate SINH_IDs, deduplicate/aggregate first to avoid row multiplication.



def normalize_colname(name: str) -> str:
    """
    Keep original casing, but:
      - trim
      - collapse any whitespace runs to underscores
    e.g. "Total Credits Block A" -> "Total_Credits_Block_A"
    """
    s = str(name).strip()
    s = re.sub(r"\s+", "_", s)
    return s

def unify_key_to_upper(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure the primary key column is exactly 'SINH_ID'.
    Handles variants like 'sinh_d' or 'sinh_id' in any casing.
    """
    lower_map = {c.lower(): c for c in df.columns}
    if "sinh_id" in lower_map:
        return df.rename(columns={lower_map["sinh_id"]: "SINH_ID"})
    if "sinh_d" in lower_map:   # some files may have this variant
        return df.rename(columns={lower_map["sinh_d"]: "SINH_ID"})
    return df  # already has SINH_ID

def prefix_nonkey_columns(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    Add '<prefix>_' to every column EXCEPT the key 'SINH_ID'.
    """
    ren = {c: f"{prefix}_{c}" for c in df.columns if c != "SINH_ID"}
    return df.rename(columns=ren)

def prepare_df(df: pd.DataFrame, df_name: str) -> pd.DataFrame:
    """
    - Unify key name to 'SINH_ID'
    - Normalize column names (spaces -> underscores)
    - Prefix non-key columns with the first token of df_name (e.g., 'sdo1')
    """
    out = unify_key_to_upper(df)
    out = out.rename(columns=lambda c: normalize_colname(c))
    prefix = df_name.split("_", 1)[0]  # first word of the df name
    out = prefix_nonkey_columns(out, prefix)
    return out

# --- clean & prefix each table --------------------------------------------

sdo1_clean = prepare_df(sdo1_characteristics, "sdo1_characteristics")
sdo2_clean = prepare_df(sdo2_skc,             "sdo2_skc")
sdo5_clean = prepare_df(sdo5_degree,          "sdo5_degree")
sdo6_clean = prepare_df(sdo6_course_results,  "sdo6_course_results")

# (Optional) sanity checks
# print(sdo1_clean.columns.tolist())
# print(sdo2_clean.columns.tolist())
# ...

# --- combine on SINH_ID ----------------------------------------------------
# Choose sdo1 as the base
data = (
    sdo1_clean
    .merge(sdo2_clean, on="SINH_ID", how="left")
    .merge(sdo5_clean, on="SINH_ID", how="left")
    .merge(sdo6_clean, on="SINH_ID", how="left")
)

# 'data' is your final merged DataFrame
print("Final shape:", data.shape)


In [ ]:
data

----------------- Save combined data frame before further cleaning -----------------------

In [ ]:
OUT_DIR = Path(os.getenv("OUTPUTS_DIR")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "v2_combined.csv"
data.to_csv(out_path, index=False)

print("Wrote:", out_path)

In [ ]:
data.columns

In [ ]:
# Drop unnecessary columns from the combined DataFrame data
data = data.drop(
    columns=[
        "sdo1_d_student_id",
        "sdo5_degree_code_letters",
        "sdo5_degree_code"
    ],
    errors="ignore"
)


In [ ]:
# Ensure no two or more columns have the same values
identical_columns = []
columns = data.columns.tolist()

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1, col2 = columns[i], columns[j]
        if data[col1].equals(data[col2]):
            identical_columns.append((col1, col2))

if identical_columns:
    print("Identical columns found:")
    for col_pair in identical_columns:
        print(col_pair)
else:
    print("No identical columns found.")

In [ ]:
# Delete the duplicate (s)
del data['sdo5_drop_out_without_degree']

In [ ]:
# Check for columns with only one unique value
single_value_cols = [col for col in data.columns if data[col].nunique() == 1]

print("Columns with only one unique value:", single_value_cols)


In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
#pd.set_option('display.expand_frame_repr', False)  


In [ ]:
# Print value counts for all columns
for col in data.columns:
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")

In [ ]:
pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
pd.set_option('display.expand_frame_repr', False)  

In [ ]:
data.columns

In [ ]:
# Convert date-related columns to datetime
date_cols = [
    "sdo1_date_of_birth",
    "sdo2_SKC_DATUM",
    "sdo5_enrollment_date",
    "sdo5_degree_start_date"
]

for col in date_cols:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors="coerce")  # invalid dates -> NaT


In [ ]:
# Print value counts only for object (categorical) columns, excluding skip_cols and datetime columns
# Columns to skip manually
skip_cols = ["SINH_ID", "sdo5_POSTCODE"]

for col in data.select_dtypes(include=["object"]).columns:
    if col in skip_cols or col in data.select_dtypes(include=["datetime"]).columns:
        continue
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")


Questions: 

What is the difference between sdo1_nationality and sdo5_LAND columns
What does 0 in sdo1_gender column means

In [ ]:
# Delete sdo5_LAND for now
del data['sdo5_LAND']

---------------------------------Handle categories in sdo1_nationality column ------------------------------------------------------

In [ ]:
# Clean categorical column (sdo1_nationality):
# - Replace unknown codes ("XXX", "XX") with "Unknown"
# - Group all categories with <1% frequency into "Other"
# - Keep only dominant categories (e.g., "NL") intact


col = "sdo1_nationality"
threshold = 0.01  # 1% cutoff

# Step 1: explicitly map special unknown codes
data[col] = data[col].replace(["XXX", "XX"], "Unknown")

# Step 2: group rare categories into "Other"
freq = data[col].value_counts(normalize=True)
rare_cats = freq[freq < threshold].index
data[col] = data[col].replace(rare_cats, "Other")

# Quick check
print(data["sdo1_nationality"].value_counts())             # raw counts
print(data["sdo1_nationality"].value_counts(normalize=True))  # proportions


-------------------------------------- Handle categories in sdo2_SKC_RESULT Column -------------------------------------------

In [ ]:
# Clean categorical column (sdo2_SKC_RESULT):
# - Replace categories with fewer than 100 occurrences with "Other"

col = "sdo2_SKC_RESULT"
threshold = 100

# Find categories with counts below the threshold
value_counts = data[col].value_counts()
rare_cats = value_counts[value_counts < threshold].index

# Replace them with "Other"
data[col] = data[col].replace(rare_cats, "Other")

# Quick check
print(data[col].value_counts(dropna=False))

-------------------------------------- Handle categories in sdo2_SKC_RESULT Column -------------------------------------------

In [ ]:
# Clean categorical values in sdo2_ADVIES_DEF:
# - Remove the word "met"
# - Replace spaces between remaining words with underscores
# - Preserve NaN values and original capitalization

col = "sdo2_ADVIES_DEF"

# Apply cleaning only on non-null values
data[col] = data[col].dropna().apply(
    lambda x: "_".join([w for w in x.split() if w.lower() != "met"])
)

# Quick check
print(data[col].value_counts())


-------------------------------------- Handle categories in sdo5_degree Column -------------------------------------------

In [ ]:
# Clean categories in sdo5_degree:
# - Keep these exact mappings:
#     "B Education in Primary Schools (age 4 - 12) (day)"     -> "Primary_Education_Day"
#     "B Education in Primary Schools (age 4 - 12) (ALPO)"    -> "Primary_Education_ALPO"
#     "B Education in Primary Schools (age 4 - 12) (Regular)" -> "Primary_Education_Regular"
#     "B Arts Therapies (Music Therapy)"                      -> "Music_Therapy"
#     "B Arts Therapies (Drama Therapy)"                      -> "Drama_Therapy"
#     "B Arts Therapies (Art Therapy)"                        -> "Art_Therapy"
# - Remove leading degree prefixes: "B " and "Bachelor of "
# - Correct known typos (e.g., "Chemics" -> "Chemistry")
# - Remove prepositions between program names: "and", "with", "in" (case-insensitive)
# - Remove commas and ampersands
# - Normalize whitespace and apply Title Case
# - Convert spaces between words into underscores at the end
# - Preserve "Teacher" as-is
# - Map NaN to "Unknown"

col = "sdo5_degree"

def clean_degree(value):
    if pd.isna(value):
        return "Unknown"
    
    v = str(value).strip()

    # ---- Exact mappings to keep as specified ----
    exact_map = {
        "B Education in Primary Schools (age 4 - 12) (day)": "Primary_Education_Day",
        "B Education in Primary Schools (age 4 - 12) (ALPO)": "Primary_Education_ALPO",
        "B Education in Primary Schools (age 4 - 12) (Regular)": "Primary_Education_Regular",
        "B Arts Therapies (Music Therapy)": "Music_Therapy",
        "B Arts Therapies (Drama Therapy)": "Drama_Therapy",
        "B Arts Therapies (Art Therapy)": "Art_Therapy",
    }
    if v in exact_map:
        return exact_map[v]

    # ---- Remove prefixes ----
    v = re.sub(r"^\s*B\s+", "", v)
    v = re.sub(r"^\s*Bachelor of\s+", "", v, flags=re.IGNORECASE)

    # ---- Fix known inconsistencies ----
    v = v.replace("Chemics", "Chemistry")

    # ---- Remove prepositions and punctuation ----
    v = v.replace("&", " ")
    v = re.sub(r"\b(?:and|with|in)\b", "", v, flags=re.IGNORECASE)
    v = v.replace(",", " ")

    # ---- Normalize whitespace and casing ----
    v = re.sub(r"\s+", " ", v).strip()
    v = v.title()

    # ---- Replace spaces with underscores ----
    v = v.replace(" ", "_")

    return v

# Apply cleaning directly to the original column
data[col] = data[col].apply(clean_degree)

# Optional check
print(data[col].value_counts())


In [ ]:
# Save version 2 data here to later combine it with version 1
data.to_csv(os.path.join("..", "data", "processed", "dropout_version2_data.csv"), header=True)

# Feature Engineering

In [ ]:
data.columns

In [ ]:
def add_group_size_column(data, group_column, new_column):
    """
    Adds a new column to the DataFrame that contains the count of each group based on the `group_column`.
    
    Parameters:
        data (DataFrame): The DataFrame where the new column will be added.
        group_column (str): The name of the column to group by.
        new_column (str): The name of the new column that will store the group sizes.
        
    Returns:
        DataFrame: The DataFrame with the new column added.
    """
    # Create a dictionary of counts for each group
    group_size = data[group_column].value_counts().to_dict()

    # Map the counts back to the dataframe as a new column
    data[new_column] = data[group_column].map(group_size)
    
    # Optionally drop the group column (if needed)
    del data[group_column]
    
    return data
    
# Example usage
data = add_group_size_column(data, 'Study_Prog_Group_ID', 'Study_Prog_Group_Size')

# Check if the new column is created and see its unique values
print(data['Study_Prog_Group_Size'].unique())


Engineer Prior_Edu_Type 

In [ ]:
print(data['Prior_Edu_Type'].value_counts())

In [ ]:
def clean_prior_edu_type(data, threshold=100, replacement="OVERIG"):
    """
    Replaces values in the 'Prior_Edu_Type' column that appear less than a specified threshold with a replacement value.
    
    Parameters:
        data (DataFrame): The DataFrame where the replacement will be applied.
        threshold (int): The threshold for the minimum count, below which values will be replaced.
        replacement (str): The value that will replace the smaller counts.
        
    Returns:
        DataFrame: The DataFrame with the modified 'Prior_Edu_Type' column.
    """
    # Get the value counts for the 'Prior_Edu_Type' column
    value_counts = data['Prior_Edu_Type'].value_counts()
    
    # Identify values with counts less than the threshold
    small_values = value_counts[value_counts < threshold].index
    
    # Replace values with counts less than the threshold with 'Others'
    data['Prior_Edu_Type'] = data['Prior_Edu_Type'].replace(small_values, replacement)
    
    return data
# Call the function to clean the 'Prior_Edu_Type' column
data = clean_prior_edu_type(data, threshold=100, replacement="OVERIG")

# Check the updated value counts
print(data['Prior_Edu_Type'].value_counts())


Engineer Prior_Edu_School_Name 

In [ ]:
print(data['Prior_Edu_School_Name'].value_counts())

In [ ]:
# Looking at how clumsy the sentences look like, I decided to use the Prior_Edu_Postcode column instead, later the postcode can be used to match the school name

Engineer Prior_Edu_Postcode

In [ ]:
print(data['Prior_Edu_Postcode'].value_counts())

In [ ]:
data.info()

In [ ]:
# Function to clean Prior_Edu_Postcode
def clean_prior_edu_postcode(data, threshold=50, replacement="Others"):
    """
    Cleans the 'Prior_Edu_Postcode' column by:
    - Converting values to strings and stripping spaces
    - Replacing any zero or asterisk values (even with spaces) with '0000_ABROAD'
    - Adding underscores between words in postcode strings
    - Replacing rare postcodes (frequency < threshold) with 'Others'
    """
    # Convert to string and strip
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].astype(str).str.strip()
    
    # Replace zeros or asterisks with '0000_ABROAD' using regex
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].replace(
        to_replace=[r'^0+$', r'^\*+$'], value='0000_ABROAD', regex=True
    )
    
    # Add underscores between words
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].apply(lambda x: "_".join(x.split()))
    
    # Replace rare values with 'Others'
    value_counts = data['Prior_Edu_Postcode'].value_counts()
    small_values = value_counts[value_counts < threshold].index
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].replace(small_values, replacement)
    
    return data

# ✅ Validator function
def validate_prior_edu_postcode(data):
    """
    Check for leftover zero, asterisk, or trailing whitespace issues in Prior_Edu_Postcode.
    """
    remaining_issues = data['Prior_Edu_Postcode'][data['Prior_Edu_Postcode'].str.match(r'^(0+|\*+)$')]
    if remaining_issues.empty:
        print("✅ No '0' or '*' values remain in Prior_Edu_Postcode!")
    else:
        print("⚠️ Still found problematic values:")
        print(remaining_issues.value_counts())
    
    # Check for trailing spaces (should not happen if cleaned, but good check)
    trailing_space_issues = data['Prior_Edu_Postcode'][data['Prior_Edu_Postcode'].str.contains(r'\s$')]
    if trailing_space_issues.empty:
        print("✅ No trailing spaces found in Prior_Edu_Postcode.")
    else:
        print("⚠️ Found trailing spaces in:")
        print(trailing_space_issues.unique())

# ✅ Example Usage:
data = clean_prior_edu_postcode(data)   # Clean the column
validate_prior_edu_postcode(data)       # Validate if everything is fixed
data['Prior_Edu_Postcode'].value_counts()  # Check distribution if desired

In [ ]:
data.columns

In [ ]:
print(data['Prior_Edu_School_Location'].value_counts())

In [ ]:
# Function to clean Prior_Edu_School_Location
def clean_prior_edu_school_location(data, threshold=50, abroad_replacement="0000_ABROAD", rare_replacement="Others"):
    """
    Cleans the 'Prior_Edu_School_Location' column by:
    - Stripping spaces
    - Replacing values '0', '*', or '-1' with abroad_replacement
    - Adding underscores between words
    - Replacing rare values (below threshold) with rare_replacement
    """
    # Strip spaces
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].astype(str).str.strip()
    
    # Replace '0', '*', or '-1' (even if they appear alone) with abroad_replacement
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].replace(
        to_replace=[r'^0+$', r'^\*+$', r'^-1$'], value=abroad_replacement, regex=True
    )
    
    # Add underscores between words
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].apply(lambda x: "_".join(x.split()))
    
    # Replace values appearing less than threshold times with rare_replacement
    value_counts = data['Prior_Edu_School_Location'].value_counts()
    small_values = value_counts[value_counts < threshold].index
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].replace(small_values, rare_replacement)
    
    return data

def validate_prior_edu_school_location(data):
    """
    Validates that no invalid values ('0', '*', '-1') or trailing spaces remain in 'Prior_Edu_School_Location'.
    """
    issues = data['Prior_Edu_School_Location'][data['Prior_Edu_School_Location'].str.match(r'^(0+|\*+|-1)$')]
    if issues.empty:
        print("✅ No invalid '0', '*', or '-1' values remain in Prior_Edu_School_Location.")
    else:
        print("⚠️ Found invalid values:")
        print(issues.value_counts())
    
    trailing_spaces = data['Prior_Edu_School_Location'][data['Prior_Edu_School_Location'].str.contains(r'\s$')]
    if trailing_spaces.empty:
        print("✅ No trailing spaces found.")
    else:
        print("⚠️ Found trailing spaces in:")
        print(trailing_spaces.unique())

# ✅ Usage:
data = clean_prior_edu_school_location(data)
validate_prior_edu_school_location(data)
data['Prior_Edu_School_Location'].value_counts()

Engineer Prior_Edu_Country

In [ ]:
data['Prior_Edu_Country'].value_counts()

In [ ]:
def clean_prior_edu_country(data, threshold=50, abroad_replacement="outside_NL", rare_replacement="Others"):
    """
    Cleans the 'Prior_Edu_Country' column by:
    - Stripping spaces
    - Replacing '0' and '-1' with abroad_replacement
    - Adding underscores between words
    - Replacing rare values (below threshold) with rare_replacement
    """
    # Strip spaces
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].astype(str).str.strip()
    
    # Replace '0' and '-1' with abroad_replacement
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].replace(
        to_replace=[r'^0+$', r'^-1$'], value=abroad_replacement, regex=True
    )
    
    # Add underscores between words
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].apply(lambda x: "_".join(x.split()))
    
    # Replace rare values
    value_counts = data['Prior_Edu_Country'].value_counts()
    small_values = value_counts[value_counts < threshold].index
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].replace(small_values, rare_replacement)
    
    return data

def validate_prior_edu_country(data):
    """
    Validates that no invalid values ('0' or '-1') or trailing spaces remain in 'Prior_Edu_Country'.
    """
    issues = data['Prior_Edu_Country'][data['Prior_Edu_Country'].str.match(r'^(0+|-1)$')]
    if issues.empty:
        print("✅ No invalid '0' or '-1' values remain in Prior_Edu_Country.")
    else:
        print("⚠️ Found invalid values:")
        print(issues.value_counts())
    
    trailing_spaces = data['Prior_Edu_Country'][data['Prior_Edu_Country'].str.contains(r'\s$')]
    if trailing_spaces.empty:
        print("✅ No trailing spaces found.")
    else:
        print("⚠️ Found trailing spaces in:")
        print(trailing_spaces.unique())

# ✅ Usage:
data = clean_prior_edu_country(data)
validate_prior_edu_country(data)
data['Prior_Edu_Country'].value_counts()

In [ ]:
data.columns

In [ ]:
df1 = data
data = df1

# Feature Extraction

In [ ]:
# Create Student_Enrollment_Gap column from Prior_Edu_Exam_Date and Student_Start_Date

# Ensure the date columns are in datetime format
data['Student_Start_Date'] = pd.to_datetime(data['Student_Start_Date'], errors='coerce')
data['Prior_Edu_Exam_Date'] = pd.to_datetime(data['Prior_Edu_Exam_Date'], errors='coerce')

# Calculate the gap in days, then convert it to weeks and round down the result
data['Student_Enrollment_Gap'] = np.floor((data['Student_Start_Date'] - data['Prior_Edu_Exam_Date']).dt.days / 7)

# Optional: To handle cases where the date conversion fails (NaT), you can fill them with a value, e.g., 0 or another placeholder
data['Student_Enrollment_Gap'].fillna(0, inplace=True)

# Check the result
data[['Student_Start_Date', 'Prior_Edu_Exam_Date', 'Student_Enrollment_Gap']].head()

In [ ]:
# Could negative values indicate that some students starting in HU before their Prior_Edu_Exam_Date? Could this be true or its not possible.
data['Student_Enrollment_Gap'].value_counts()

In [ ]:
pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
pd.set_option('display.expand_frame_repr', False)  

In [ ]:
# Could negative values indicate that some students starting in HU before their Prior_Edu_Exam_Date? Could this be true or its not possible.
data['Student_Enrollment_Gap'].value_counts()

In [ ]:
print(data['Student_Enrollment_Gap'].nlargest(100))


In [ ]:
# Clean and preprocess the Student_Enrollment_Gap column by removing negative signs, 
# capping values over 522 weeks (10 years), and imputing them using KNN (K=5) to ensure 
# all values stay within a valid range.

# Step 1: Remove minus signs and convert to numeric
data['Student_Enrollment_Gap'] = data['Student_Enrollment_Gap'].astype(str).str.replace('-', '', regex=False)
data['Student_Enrollment_Gap'] = pd.to_numeric(data['Student_Enrollment_Gap'], errors='coerce')

# Step 2: Replace values > 522 with NaN (so we can impute them)
data.loc[data['Student_Enrollment_Gap'] > 522, 'Student_Enrollment_Gap'] = np.nan

# Step 3: Use KNN Imputer with K=5
imputer = KNNImputer(n_neighbors=5)
data[['Student_Enrollment_Gap']] = imputer.fit_transform(data[['Student_Enrollment_Gap']])

# Step 4: Final check to make sure no value exceeds 522
data.loc[data['Student_Enrollment_Gap'] > 522, 'Student_Enrollment_Gap'] = 522


In [ ]:
data['Student_Enrollment_Gap'].value_counts()

In [ ]:
print(data['Student_Enrollment_Gap'].nlargest(10))

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
data.columns

Create Exit_Status common from 'Graduating', 'Dropping', and 'Switching'

In [ ]:
data.info()

In [ ]:
data[['Graduating']].value_counts()

In [ ]:
data[['Dropping']].value_counts()

In [ ]:
data[['Switching']].value_counts()

In [ ]:
# Step 1: Create the 'Exit_Status' column and set default to 'Dropping'
data['Exit_Status'] = 'dropping'  

# Step 2: Assign 'Graduating' where graduating == 1
data.loc[data['Graduating'] == 1, 'Exit_Status'] = 'graduating'  

# Step 3: Assign 'switching' where switching == 1
data.loc[data['Switching'] == 1, 'Exit_Status'] = 'switching'  

# Step 4: Drop the original columns since they are now redundant
data.drop(columns=['Graduating', 'Dropping', 'Switching'], inplace=True)  

# Step 5: Verify the results
print(data['Exit_Status'].value_counts())


# Sample Data by undersampling

In [ ]:
# Shuffle the dataset first
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Find the minimum count among categories
min_count = data["Exit_Status"].value_counts().min()

# Perform undersampling after shuffling
undersampled_data = (
    data.groupby("Exit_Status")
    .apply(lambda x: x.sample(n=min_count, random_state=42))
    .reset_index(drop=True)
)

# Display the new counts
print(undersampled_data["Exit_Status"].value_counts())


In [ ]:
data = undersampled_data

In [ ]:
data.columns

In [ ]:
data.Study_Pro_Start_Year.unique()

In [ ]:
print(data['Study_Pro_Start_Year'].value_counts())

In [ ]:
data.to_csv(os.path.join("..", "data", "processed", "EDA.csv"), header=True)

In [ ]:
# Drop these columns because they do not add more value to the analysis

data.drop(columns=['Student_num', 
                   'Study_Pro_Start_Year',
                   'Student_Start_Date', 
                   'Student_End_Date', 
                   'Prior_Edu_Exam_Date',
                   'Prior_Edu_School_Name'], 
                   inplace=True)
data.columns

In [ ]:
data.to_csv(os.path.join("..", "data", "processed", "features_engineered.csv"), header=True)

In [ ]:
file_path = os.path.join("..", "data", "processed", "features_engineered.csv")

# Read CSV with index_col=0 to avoid unnamed column issues
df2 = pd.read_csv(file_path, index_col=0)

# Display the DataFrame
data = df2
data

# Feature Extraction

In [ ]:
data.info()

In [ ]:
print(data['Study_Prog_Exam_Completed'].value_counts())

In [ ]:
print(data['Prior_Edu_Country'].value_counts())

In [ ]:
# Define a mapping for Prior_Edu_Country
prior_edu_country_mapping = {
    'outside_NL': 0,
    'NL': 1
}

# Apply the mapping to replace categorical values with numbers
data['Prior_Edu_Country'] = data['Prior_Edu_Country'].map(prior_edu_country_mapping)

# Show the first few rows to verify
print(data['Prior_Edu_Country'].value_counts())


In [ ]:
data.info()

In [ ]:
# One-hot-encoded all columns with object data type

# Step 1: Identify all object (categorical) columns except 'Exit_Status'
categorical_cols = data.select_dtypes(include=['object']).columns  
categorical_cols = categorical_cols[categorical_cols != 'Exit_Status']  # Exclude 'Exit_Status'

# Step 2: Perform one-hot encoding with column names as prefixes
data = pd.get_dummies(data, columns=categorical_cols, prefix=categorical_cols, dtype=int)  

# Step 3: Check the transformed DataFrame  
data  



In [ ]:
data.to_csv(os.path.join("..", "data", "processed", "feature_select.csv"), header=True)